In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchaudio.transforms import Resample
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import FullSubPathExtension 

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3

import matplotlib.pyplot as plt

In [2]:
CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
N_FFTS = 512
HOP_LENGTH = 256
HID_SIZE = 32
SR = 16_000

# N_FFTS = 1024
# HOP_LENGTH = 512
# HID_SIZE = 64
# SR = 48_000

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [4]:
from src.fspen_configs import TrainConfig, TrainConfig_48khz

configs = TrainConfig()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_OG_rirs#3.pt"), map_location="cpu",  weights_only=False)

In [5]:
fspen.load_state_dict(state_d["model_state_dict"])

<All keys matched successfully>

In [6]:
state_d["plots"]["val loss"]

[0.007393902597519068,
 0.0062623997577107866,
 0.005685537337110593,
 0.006107498049879303,
 0.005906842935543794,
 0.005777475227100344,
 0.005777187072313749,
 0.005557203199714422,
 0.0056565622619998,
 0.005278282986882214,
 0.005399828454336295,
 0.0055682605239920895,
 0.0056915591972378585,
 0.005535333793467054,
 0.0058881825086875604,
 0.005538854676370437,
 0.005461270204530313,
 0.005850522581917735,
 0.005735508488634458,
 0.005787122797650786,
 0.00569221515280123,
 0.005931894187457287,
 0.0057450695894658566,
 0.005593514768406749,
 0.005841810739814089,
 0.005865683331369207,
 0.005593133015701404,
 0.005621061320058429,
 0.005609401036053896,
 0.005375984948701584,
 0.0056640760650715,
 0.005635936672870929,
 0.0055659727838176945,
 0.0055029778024898125,
 0.005549333452318723,
 0.005313411265468368,
 0.0052891544854411715,
 0.00519714741788518,
 0.005259223658448229,
 0.005236916745511385,
 0.005091544509363862,
 0.005080541962972627,
 0.005139546343483604,
 0.005166

In [7]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [8]:
AUDIO_PATH = "/home/zakhar/ems_dereverb/input_sig_full.wav"

In [9]:
signal, signal_sr = torchaudio.load(AUDIO_PATH)

input_signal, _ = SignalDataset.normalize_audio(signal)

if signal_sr != SR:
    resampler = Resample(signal_sr, SR)
    input_signal = resampler(input_signal)

In [10]:
Audio(input_signal, rate=SR)

In [11]:
window = vorbis_window(N_FFTS)

spec = torch.stft(
            input_signal,
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

output, _ = model_eval(fspen, spec, configs, DEVICE, hid_size=32)

out_wave = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                       window=window,
                       # onesided=True,
                       return_complex=False,
                       normalized=True,
                       center=True)

out_wave = out_wave.reshape(-1)

In [12]:
from scipy.io.wavfile import write

write(AUDIO_PATH[:-4] + "_og_16khz.wav", SR, out_wave.cpu().detach().numpy())

In [13]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [14]:
from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)

In [15]:
target, signal_sr = torchaudio.load("/home/zakhar/ems_dereverb/gt_full.wav")

if signal_sr != SR:
    resampler = Resample(signal_sr, SR)
    target = resampler(target)

target, _ = SignalDataset.normalize_audio(target)

In [16]:
min_l = min(out_wave.shape[-1], input_signal.shape[-1])
nisqa_score_in, _, _ = process(input_signal.unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
nisqa_score_out, _, _ = process(out_wave.unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
print("NISQA in:", nisqa_score_in)
print("NISQA out:", nisqa_score_out)

stoi_score_in = -stoi(input_signal[..., :min_l], target[..., :min_l])
stoi_score_out = -stoi(out_wave[..., :min_l].unsqueeze(0), target[..., :min_l])
print(f"STOI in:", stoi_score_in.item())
print(f"STOI out:", stoi_score_out.item())

# resampler = Resample(SR, 16_000)
# signal = resampler(signal.cpu())
# out_wave = resampler(out_wave.cpu())
# target = resampler(target)
# target = resampler(target.cpu()).cuda()
# min_l = min(output.shape[-1], target.shape[-1])

srmr_score_in = srmr(input_signal.detach())
srmr_score_out = srmr(out_wave.detach())
srmr_score_target = srmr(target)
print(f"SRMR in: {srmr_score_in:.2f}")
print(f"SRMR out: {srmr_score_out:.2f}")
print(f"SRMR target: {srmr_score_target:.2f}")

min_l = min(out_wave.shape[-1], input_signal.shape[-1], target.shape[-1])

# print(out_wave.shape, target.shape, input_signal.shape, min_l)
pesq_score_in = pesq(out_wave[..., :min_l], target[0, :min_l])
pesq_score_out = pesq(input_signal[0, :min_l], target[0, :min_l])
pesq_score_targer = pesq(target[0, :min_l], target[0, :min_l])

print(f"PESQ in: {pesq_score_in:.2f}")
print(f"PESQ out: {pesq_score_out:.2f}")
print(f"PESQ target: {pesq_score_targer:.2f}")

NISQA in: tensor([[1.643, 2.198, 3.117, 2.273, 2.250]])
NISQA out: tensor([[1.329, 2.597, 1.759, 2.268, 2.604]])
STOI in: 0.7002800703048706
STOI out: 0.674892783164978
SRMR in: 9.72
SRMR out: 10.44
SRMR target: 10.90
PESQ in: 2.52
PESQ out: 2.06
PESQ target: 4.64
